# Latent Bayesian Hyperparameter Optimization

Select one dataset, define a mixed hyperparameter search space, and optimize SCALP graph parameters with Bayesian optimization in a low-dimensional latent space.

In [1]:
import sys
import subprocess


def ensure_bo_dependencies():
    try:
        import botorch  # noqa: F401
        import gpytorch  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "..[bo]"])


ensure_bo_dependencies()


In [2]:
import numpy as np

from scalp_lite.metrics import score_embedding
from scalp_lite.notebook_utils import dataset_config, load_preprocessed_data, make_estimator
from scalp_lite.optimization import latent_bayesopt


selected_dataset = "pancreas"
dataset = dataset_config(selected_dataset)

# Keep objective evaluations manageable. Increase after validating the search setup.
preprocess_overrides = {"max_cells": 2000}
random_state = 0
estimator = make_estimator(dataset, n_components=100, random_state=random_state)
adata = load_preprocessed_data(estimator, dataset, **preprocess_overrides)
adata


AnnData object with n_obs × n_vars = 2000 × 1000
    obs: 'batch', 'study', 'cell_type', 'size_factors'
    obsm: 'X_pca'

In [3]:
base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}

search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 40]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 8]},
    "assignment_quantile": {"type": "float", "bounds": [0.05, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 30]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

base_graph_params, search_space


({'n_neighbors': 15,
  'intra_fraction': 0.5,
  'n_inter_edges': 2,
  'metric': 'euclidean',
  'assignment_quantile': 0.8,
  'hubness_correction': 'csls',
  'hubness_k': 10,
  'rank_correction': True,
  'edge_weighting': 'binary',
  'mutual_neighbors': False,
  'neighbor_mode': 'distance',
  'symmetrize': True},
 {'n_neighbors': {'type': 'int', 'bounds': [5, 40]},
  'intra_fraction': {'type': 'float', 'bounds': [0.2, 0.9]},
  'n_inter_edges': {'type': 'int', 'bounds': [1, 8]},
  'assignment_quantile': {'type': 'float', 'bounds': [0.05, 1.0]},
  'hubness_k': {'type': 'int', 'bounds': [3, 30]},
  'rank_correction': {'type': 'categorical', 'values': [False, True]},
  'edge_weighting': {'type': 'categorical', 'values': ['binary', 'distance']},
  'mutual_neighbors': {'type': 'categorical', 'values': [False, True]}})

In [4]:
def metric_value(scores, key, default=0.0):
    value = float(scores.get(key, default))
    return default if not np.isfinite(value) else value


def scalp_objective(params):
    graph_params = {**base_graph_params, **params}
    trial = adata.copy()
    try:
        trial.obsm["X_scalp"] = estimator.embed(
            trial,
            **graph_params,
            embedding_random_state=random_state,
            n_epochs=100,
        )
        scores = score_embedding(
            trial,
            embedding_key="X_scalp",
            batch_key=dataset["batch_key"],
            label_key=dataset["label_key"] if dataset["label_key"] in trial.obs else None,
        ).iloc[0]
    except Exception as exc:
        print(f"failed params={graph_params}: {exc}")
        return -1e9

    label_agreement = metric_value(scores, "knn_label_agreement")
    batch_mixing = metric_value(scores, "batch_mixing")
    graph_density = metric_value(scores, "graph_density")
    # Maximize biological coherence and batch mixing; tune weights for your target use case.
    return float(label_agreement + 0.25 * batch_mixing - 0.05 * graph_density)


In [ ]:
# Start with PCA for a cheap sanity check. Switch to "gplvm" for nonlinear latent optimization.
bo_result = latent_bayesopt(
    scalp_objective,
    search_space,
    n_initial=20,
    latent_dim=3,
    n_iterations=15,
    embedding_model="pca",
    acquisition="ei",
    batch_size=1,
    random_state=random_state,
    invalid_score=-1e9,
)

bo_result["best_params"], bo_result["best_score"]


/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/botorch/models/utils/assorted.py:276: InputDataWarning: Data (input features) is not contained to the unit cube. Please consider min-max scaling the input data.
  check_min_max_scaling(
/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/botorch/models/utils/assorted.py:276: InputDataWarning: Data (input features) is not contained to the unit cube.

({'n_neighbors': 33,
  'intra_fraction': 0.8999999999999999,
  'n_inter_edges': 8,
  'assignment_quantile': 1.0,
  'hubness_k': 14,
  'rank_correction': True,
  'edge_weighting': 'distance',
  'mutual_neighbors': True},
 0.9437500000000001)

In [6]:
bo_result["history"].sort_values("score", ascending=False).head(10)


,iteration,phase,score,n_neighbors,intra_fraction,n_inter_edges,assignment_quantile,hubness_k,rank_correction,edge_weighting,mutual_neighbors
7,2,bo,0.943750,33,0.900000,8,1.000000,14,True,distance,True
8,3,bo,0.942625,36,0.900000,8,1.000000,11,False,distance,True
9,4,bo,0.941525,40,0.900000,8,1.000000,9,False,distance,True
6,1,bo,0.940400,29,0.841764,7,1.000000,16,True,distance,True
5,0,initial,0.930150,32,0.898047,4,0.981794,13,True,distance,True
1,0,initial,0.924450,28,0.624645,8,0.743022,20,True,distance,True
3,0,initial,0.797600,35,0.804225,2,0.564388,5,False,binary,False
2,0,initial,0.765725,14,0.201917,7,0.864534,18,False,distance,True
0,0,initial,0.718275,35,0.388851,6,0.088925,5,False,binary,True
4,0,initial,0.357625,19,0.286998,1,0.687093,17,True,binary,True
